In this Notebook we look at the relationship between the mean field curve and the (normalised) Hamming weight of the local update rule. These are intimitely connected, because

1. The Hamming weight is proportional to the number of neighbourhood states that map to a living node.
2. The mean field curve looks at the average density over the entire network, and considers what density this should map to.

We will also briefly consider the mean field parameters.

The expected value $\langle s_i \rangle$ of a single randomly chosen node is the same as the state average over the entire network, which we call $\rho$. This expected value, and therefore the state average, evolves as follows:

$$
\langle s_i(t=1) \rangle = \sum_{s\in\{0, 1\}}\sum_{q=0}^{k_i} P(q ~|~ k_i, \rho(t=0)) P(s ~|~ \rho(t=0))~\phi(s, q/k_i).
$$

Here $\rho(t=0)$ is the average state of the entire network, and ...
- $P(q ~|~ k_i, \rho(t=0))$ is the probability of encountering a neighbourhood that sums to $q$, given that the node has degree $k$ and that the current average density in the network is $\rho(t=0)$.
- $P(s ~|~ \rho(t=0))$ is the probability of encountering a node that is currently in state $s$, given that the current average density in the network is $\rho(t=0)$.

Let's rename $\langle s_i(t=0) \rangle = \langle s_i^0 \rangle$ and $\rho(t=1) = \rho^1$. Both probabilities are simply binomial distributions, i.e.

$$
\begin{align*}
P(q ~|~ k_i, \rho^0) &= \dbinom{k_i}{q} (\rho^0)^{q} (1-\rho^0)^{k_i-q}\\
P(s ~|~ \rho^0) &= (\rho^0)^s (1-\rho^0)^{1-s}
\end{align*}
$$

Consider three special cases. First, if $\rho^0 \to 0$, we find (with $\delta$ the Kronecker delta)
$$
\begin{align*}
\begin{cases}
P(q ~|~ k_i, \rho^0) &= \delta_{q,0}\\
P(s ~|~ \rho^0) &= \delta_{s,0}
\end{cases}, \qquad \text{so } \langle s_i^1 \rangle = \phi(0, 0)
\end{align*}
$$
If $\rho^0 \to 1$, we find
$$
\begin{align*}
\begin{cases}
P(q ~|~ k_i, \rho^0) &= \delta_{q,k_i}\\
P(s ~|~ \rho^0) &= \delta_{s,1}
\end{cases}, \qquad \text{so } \langle s_i^1 \rangle = \phi(1, k_i)
\end{align*}
$$
If $\rho^0 = 1/2$, we find
$$
\begin{align*}
\begin{cases}
P(q ~|~ k_i, \rho^0) &= \dfrac{1}{2^{k_i}}\dbinom{k_i}{q}\\
P(s ~|~ \rho^0) &= \dfrac{1}{2}
\end{cases}, \qquad \text{so } \langle s_i^1 \rangle = \text{HW}_i
\end{align*}
$$
where $\text{HW}_i$ is simply the Hamming weight for node $v_i$ with degree $k_i$. All intermediate states can be plotted quite easily.

In order to find the average response of the entire network with $N$ nodes, we must average over the nodes.
$$
\begin{align*}
\rho^1 = \frac{1}{N}\sum_{i=1}^N \langle s^1_i \rangle = \sum_{k}\sum_{s\in\{0, 1\}}\sum_{q=0}^{k} P(k) P(q ~|~ k, \rho^0) P(s ~|~ \rho^0)~\phi(s, q/k),
\end{align*}
$$
where $P(k)$ represents the node degree distribution of the network at hand, and the first sum runs over all possible degree values.

**Corrollary**

What is the slope of the curve through the origin?

This may be interesting because it mimicks the Derrida coefficient, and it tells you how, on average, an single seed will spread. It does not really have a very nice closed mathematical form, however, but let's show it anyway.

The expected value $\langle s_i^1 \rangle$ of node $v_i$ when a single node is seeded is
$$
\langle s_i^1(\rho^0=1/N) \rangle = \frac{1}{N}\left( \phi(1,0) + \sum_{v_j \in \mathcal{N}(v_i)} \phi(0, 1/k_j) + (N - k_i - 1)\phi(0,0)\right)
$$
Therefore, the slope through the origin is
$$
\frac{\langle s_i^1(\rho^0=1/N)\rangle - \langle s_i^1(\rho^0=1/N)\rangle}{1/N} = \phi(1,0) - (k_i + 1)\phi(0,0) + \sum_{v_j \in \mathcal{N}(v_i)}\phi(0, 1/k_j)
$$
The slope is therefore certainly constrained to values in $[-(k_\text{max}+1), k_\text{max}+1]$.
If we take the average over the network, we find
$$
\text{slope} = \phi(1,0) + \frac{1}{N}\sum_{i=1}^N\left[\sum_{v_j \in \mathcal{N}(v_i)}\phi(0, 1/k_j) - (k_i + 1)\phi(0,0)\right]
$$

Another special case is when the network is regular, such that $\forall i : k_i = k$. In that case
$$
\text{slope} = \phi(1,0) - \phi(0,0) + k(\phi(0, 1/k) - \phi(0,0))
$$

In [ ]:
# standard preamble for the Notebooks I use

import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

from src.automata import LLNA
from src.simulation import *
from src.analysis import hamming_weight, mean_field_dens_propagation, binary_indices
from src.library import return_life_like_dict

from scipy.stats import binom

%load_ext autoreload
%autoreload 2

In [ ]:
life_like_dict = return_life_like_dict()
life_like_dict

In [ ]:
life_like_name = "highlife" # i like: replicator, diamoeba, 34_life, 2x2
beta, sigma = life_like_dict[life_like_name]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

# this is some kind of Fourier transformation!
resolution = 9
model = LLNA(resolution, B_set, S_set, iso=True)
B_set, S_set = model.rule
degrees = [4, 8, 12]

# curve
num_dens_points = 101
current_dens = np.linspace(0,1,num_dens_points)
next_dens_list = []
for degree in degrees:
    next_dens = np.array([mean_field_dens_propagation(resolution, B_set, S_set, current_d, degree) for current_d in current_dens])
    next_dens_list.append(next_dens)

# scatter
HW_list = []
for degree in degrees:
    HW = hamming_weight(resolution, B_set, S_set, degree)
    HW_list.append(HW)
LHS = int(0 in B_set)
RHS = int(resolution-1 in S_set)

for next_dens, degree, HW in zip(next_dens_list, degrees, HW_list):
    plt.plot(current_dens, next_dens, label=f'Degree {degree}, HW {round(HW,2)}')
    # plt.scatter(0.5, HW, color='black', s=100, marker='*')
# plot average
next_dens_average = np.array(next_dens_list).mean(axis=0)
plt.plot(current_dens, next_dens_average, color='black', label="Average", lw=4, ls='-.')
# plt.plot([0,1], [0,1], 'k--')
HW_average = next_dens_average[num_dens_points//2]
plt.plot([0, .5], [HW_average, HW_average], 'k--', label="Average Hamming weight")
plt.scatter(0, LHS, color='black', s=100, marker='+')
plt.scatter(1, RHS, color='black', s=100, marker='+')

fontsize=20

plt.legend()
plt.xlabel("Original average density $\\rho^0$", fontsize=fontsize)
plt.ylabel("Next average density $\\rho^1$", fontsize=fontsize)
plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])
plt.title(f"{model.__str__(latex=True)}, {len(degrees)} different degrees", size=fontsize+4)

This is pretty neat! Now make sure that this corresponds to a realistic situation.

In [ ]:
def init_config_with_dens(N, dens):
    # defines a random initial configuration with a fixed state density
    s0 = np.zeros(N, dtype=float)
    s0[:np.round(dens*N).astype(int)] = 1.
    np.random.shuffle(s0)
    return s0

resolution = 9
model = LLNA(resolution, iso=True)
B_set, S_set = model.rule

N = 200
N_dens = 201
current_dens = np.linspace(0,1,N_dens)

samples_per_dens = 50
current_dens_simulated = np.repeat(current_dens, samples_per_dens)
init_configs = np.array([init_config_with_dens(N, dens) for dens in current_dens for _ in range(samples_per_dens)])

import igraph as ig
import matplotlib.pyplot as plt

# Parameters for the Watts-Strogatz network
k = 6  # Each node is connected to k nearest neighbors in ring topology
p = 0.5  # Rewiring probability

# Generate the Watts-Strogatz network
ws_graph = ig.Graph.Watts_Strogatz(dim=1, size=N, nei=k//2, p=p)
# ring_graph = ig.Graph.Ring(N)
# find edges
ws_graph.to_directed()
edges = tc.tensor(ws_graph.get_edgelist()).T
ws_graph.to_undirected()
# ring_graph.to_directed()
# edges = tc.tensor(ring_graph.get_edgelist()).T
# ring_graph.to_undirected()

# generate the next timestep in this graph
s = model.step(edges, tc.tensor(init_configs)).numpy()
next_dens_simulated = np.mean(s, axis=1)

# Get the degrees of all nodes
degrees = ws_graph.degree()
# degrees = ring_graph.degree()

# Calculate the degree distribution
degree_counts = np.bincount(degrees)
degree_list = np.arange(len(degree_counts))
degree_probabilities = degree_counts / sum(degree_counts)

# calculate the mean field propagation over all node degrees
next_dens_mean_field = np.sum([degree_prob*np.array([mean_field_dens_propagation(resolution, B_set, S_set, current_d, degree) for current_d in current_dens]) for degree, degree_prob in zip(degree_list, degree_probabilities)], axis=0)

# plot everything 
plt.scatter(current_dens_simulated, next_dens_simulated, s=1, alpha=0.05)
plt.plot(current_dens, next_dens_mean_field)
plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])

plt.xlabel("Original average density")
plt.ylabel("Average density in the next time step")
plt.title(f"{model.__str__(latex=True)} on Watts-Strogatz model\n$p={round(p,2)}$, $\\langle k \\rangle={k}$, $N={N}$, ", size=24)

This actually works like a charm as well! So nice :)